# QuotaGate Notebook 

This is a **reviewer-friendly notebook** derived from the much larger research notebook.

It keeps only the final reproducible line of research that matches the paper:
- **QuotaGate-U** (paced base controller)
- **QuotaGate-RawGap / older raw-gap ablation**
- **QuotaGate-EWBE**
- **exact inverse transport calibration** for the synthetic global multiplicative width-mismatch model

It removes exploratory branches that are no longer part of the final paper narrative.

## Folder layout beside this notebook

Set `PROJECT_ROOT` below to the directory that contains the run assets.

### Required folders
- `tables/`
- `records/`

These are the only required inputs for the final exact-inverse pipeline. The final scripts read `cand_sweep_pd.csv` or `cand_sweep.csv` from `tables/` and cached validation/test pickle records from `records/`. The outputs are then written into a new folder under the same root. This matches the final paper’s evaluation setup, which uses cached record files and a fixed headline operating point with synthetic width scales `{0.8, 1.0, 1.2}`.

### Optional folders for browsing older supporting evidence
- `final_submission_pack_final_gap_closer/` — older base-controller and semantic-bridge evidence
- `committee_fix_round/` — older risk-beyond-utility supporting evidence
- `expected_wbe_exact_inverse_round/` — if you already have precomputed final outputs and only want to inspect them

### Suggested layout
```text
reviewer_bundle/
  QuotaGate_reviewer_curated.ipynb
  tables/
  records/
  final_submission_pack_final_gap_closer/      # optional
  committee_fix_round/                         # optional
  expected_wbe_exact_inverse_round/            # optional, precomputed outputs
```


In [ ]:
# Initial config — replace this path with your local copy
from pathlib import Path
import os, glob, json, pickle

PROJECT_ROOT = Path('./PROJECT_FOLDER').resolve()   # <- CHANGE THIS IF NEEDED
os.environ['RUN_DIR'] = str(PROJECT_ROOT)

TABLES_DIR = PROJECT_ROOT / 'tables'
RECORDS_DIR = PROJECT_ROOT / 'records'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('TABLES_DIR   =', TABLES_DIR)
print('RECORDS_DIR  =', RECORDS_DIR)

In [ ]:
# Optional: install missing libraries in a fresh environment
# Uncomment if needed.
# !pip -q install numpy pandas matplotlib scikit-learn

In [ ]:
# Preflight check — required inputs for the final exact-inverse round
from pathlib import Path
import glob, os

assert TABLES_DIR.exists(), f'Missing required folder: {TABLES_DIR}'
assert RECORDS_DIR.exists(), f'Missing required folder: {RECORDS_DIR}'

cand_pd = TABLES_DIR / 'cand_sweep_pd.csv'
cand_plain = TABLES_DIR / 'cand_sweep.csv'
assert cand_pd.exists() or cand_plain.exists(), 'Need tables/cand_sweep_pd.csv or tables/cand_sweep.csv'

pkl_hits = sorted(glob.glob(str(RECORDS_DIR / '*__val__*.pkl'))) + sorted(glob.glob(str(RECORDS_DIR / '*__test__*.pkl'))) +            sorted(glob.glob(str(RECORDS_DIR / '*_val_*.pkl'))) + sorted(glob.glob(str(RECORDS_DIR / '*_test_*.pkl')))
assert len(pkl_hits) > 0, f'No cached val/test pickle files found in {RECORDS_DIR}'

print('Found candidate sweep file:', cand_pd if cand_pd.exists() else cand_plain)
print('Number of cached val/test record pickles:', len(pkl_hits))
print('Sample files:')
for f in pkl_hits[:6]:
    print(' -', os.path.basename(f))

## Final executable cell kept from the original notebook

This cell is the **only executable research cell retained** from the original notebook. It corresponds to the final exact-inverse line and writes the final-paper artifacts under:

```text
PROJECT_ROOT / expected_wbe_exact_inverse_round/
```

It uses only the required `tables/` and `records/` inputs and reproduces the final paper’s main setting, strict-target/runtime, validation diagnostics, and exact reciprocal recovery audit. The exact-inverse story is the final paper line: the controller keeps the same EWBE score and pacing shell, while the synthetic robustness family is repaired analytically by setting `c = 1/s`. fileciteturn29file1

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Expected-WBE RDU round with exact inverse transport calibration
===============================================================

This is the theorem-backed version of the robustness repair.

Core method
-----------
Keep the SAME QuotaGate pacing shell and the SAME expected-WBE score:
    sev = exceed * p_hat
where
    exceed = max(R_unc - B, 0)
    p_hat  = P(miss of violating slate | decision-time features)

Main setting
------------
The decisive main setting is still tuned exactly as before at:
    tau = 0.90
    K = 10
    width_scale = 1.0

Exact transport repair
----------------------
Under the synthetic global multiplicative width mismatch family,
all width dependence enters as:

    w_eff = width_scale * cal_scale * w

So if the nominal run is at (width_scale, cal_scale) = (1, 1),
exact recovery requires:

    width_scale * cal_scale = 1

Therefore this script uses the analytic inverse:
    cal_scale = 1 / width_scale

and keeps frozen:
    - lambda*
    - miss model p_hat
    - feature scales
    - severity scale

This removes the coarse-grid approximation error from the prior transport
script and directly tests the product-invariance theorem.

Notebook run
------------
    # This notebook embeds the exact-inverse round directly below.


"""

import os
import glob
import math
import pickle
import time
import shutil
import textwrap
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.model_selection import StratifiedKFold
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False


RUN_DIR = os.environ.get("RUN_DIR", str(PROJECT_ROOT))
OUT_NAME = "expected_wbe_exact_inverse_round"
TABLES_DIR = os.path.join(RUN_DIR, "tables")
RECORDS_DIR = os.path.join(RUN_DIR, "records")
OUT_DIR = os.path.join(RUN_DIR, OUT_NAME)
OUT_TABLES = os.path.join(OUT_DIR, "tables")
OUT_FIGS = os.path.join(OUT_DIR, "figs")
OUT_SNIPS = os.path.join(OUT_DIR, "snippets")
os.makedirs(OUT_TABLES, exist_ok=True)
os.makedirs(OUT_FIGS, exist_ok=True)
os.makedirs(OUT_SNIPS, exist_ok=True)

MAIN_TAU = 0.90
MAIN_K = 10
ROBUST_TAUS = [0.90, 0.95]
ROBUST_WIDTH_SCALES = [0.80, 1.00, 1.20]

CTRL_SEEDS = [2025, 2026, 2027]
CV_SEEDS = [2025, 2026, 2027, 2028, 2029]
N_BOOT = 500
EPS = 1e-12
WARMUP = 20

LAMBDA_GRID = [0.0, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]
PENALTY_LAM_GRID = [0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0, 16.0]
ADAPTIVE_ETA_GRID = [0.10, 0.25, 0.50, 1.0, 2.0, 4.0]
GAMMA_GRID = [0.0, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 1.0, 2.0, 4.0, 8.0, 16.0]

FORCE_ROBUSTNESS = True

cand_path = (
    os.path.join(TABLES_DIR, "cand_sweep_pd.csv")
    if os.path.exists(os.path.join(TABLES_DIR, "cand_sweep_pd.csv"))
    else os.path.join(TABLES_DIR, "cand_sweep.csv")
)
cand_df = pd.read_csv(cand_path)


def topk_idx(scores, k=10):
    scores = np.asarray(scores)
    n = scores.shape[0]
    if n <= k:
        idx = np.arange(n, dtype=int)
        return idx[np.argsort(-scores[idx])]
    idx = np.argpartition(-scores, kth=k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return idx


def risk_of_take(width_arr, take):
    return float(np.asarray(width_arr)[take].sum())


def proxy_utility(mu, take):
    return float(np.asarray(mu)[take].sum())


def hit_at_k(items_ranked, positives, k=10):
    pos = set(positives)
    return float(len(set(items_ranked[:k]) & pos) > 0)


def recall_at_k(items_ranked, positives, k=10):
    pos = set(positives)
    if len(pos) == 0:
        return 0.0
    return float(len(set(items_ranked[:k]) & pos) / len(pos))


def ndcg_at_k(items_ranked, positives, k=10):
    rel = set(positives) if isinstance(positives, (list, tuple, set, np.ndarray)) else {positives}
    dcg = 0.0
    for i, item in enumerate(items_ranked[:k]):
        if item in rel:
            dcg += 1.0 / math.log2(i + 2.0)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(i + 2.0) for i in range(ideal_hits))
    return float(dcg / idcg) if idcg > 0 else 0.0


def timed_call(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return out, (t1 - t0)


def bootstrap_delta(a, b, n_boot=500, seed=2025):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    d = a - b
    rng = np.random.default_rng(seed)
    n = len(d)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots.append(float(np.mean(d[idx])))
    lo = float(np.quantile(boots, 0.025))
    hi = float(np.quantile(boots, 0.975))
    return float(np.mean(d)), lo, hi, bool((lo > 0.0) or (hi < 0.0))


def agg_mean_std(df, group_cols, metric_cols):
    mean_df = df.groupby(group_cols, as_index=False)[metric_cols].mean()
    std_df = df.groupby(group_cols, as_index=False)[metric_cols].std().fillna(0.0)
    std_df = std_df.rename(columns={c: c + "_std" for c in metric_cols})
    return mean_df.merge(std_df, on=group_cols, how="left")


def save_df(df, name):
    path = os.path.join(OUT_TABLES, name)
    df.to_csv(path, index=False)
    print("Saved:", path, df.shape)
    return path


def observed_width(rec, width_scale=1.0, cal_scale=1.0):
    return np.asarray(rec["width"], dtype=float) * float(width_scale) * float(cal_scale)


def find_record_file(ds_name, split, cand_n):
    patterns = [
        os.path.join(RECORDS_DIR, f"{ds_name}__{split}__N{cand_n}__cap*.pkl"),
        os.path.join(RECORDS_DIR, f"{ds_name}_{split}_N{cand_n}_cap*.pkl"),
    ]
    hits = []
    for pat in patterns:
        hits.extend(sorted(glob.glob(pat)))
    hits = sorted(set(hits))
    if len(hits) == 0:
        raise FileNotFoundError(f"No record file found for {ds_name=} {split=} {cand_n=}")
    return hits[0]


def load_records(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def choose_largest_cand(ds_name):
    sub = cand_df[cand_df["dataset"] == ds_name].copy()
    return int(sub["cand_n"].astype(int).max())


def resolve_budget(records, ds_name=None, cand_n=None, k_target=10):
    B10 = None
    if len(records) and isinstance(records[0], dict) and ("B" in records[0]):
        try:
            B10 = float(records[0]["B"])
        except Exception:
            B10 = None
    if (B10 is None or B10 <= 0) and ds_name is not None:
        sub = cand_df[cand_df["dataset"] == ds_name].copy()
        if cand_n is not None and "cand_n" in sub.columns:
            sub = sub[sub["cand_n"].astype(int) == int(cand_n)].copy()
        for col in ["B", "budget", "risk_budget", "B10"]:
            if col in sub.columns:
                vals = pd.to_numeric(sub[col], errors="coerce")
                vals = vals[np.isfinite(vals) & (vals > 0)]
                if len(vals):
                    B10 = float(vals.iloc[0]); break
    if B10 is None or B10 <= 0:
        raise ValueError(f"Could not resolve positive B for dataset={ds_name}, cand_n={cand_n}")
    return B10 * (k_target / 10.0), B10


def metrics_from_take(rec, take, B_true, k=10):
    items_take = np.asarray(rec["items"])[take].tolist()
    risk_true = risk_of_take(np.asarray(rec["width"]), take)
    hit = hit_at_k(items_take, rec["pos"], k=k)
    miss = 1.0 - hit
    return {
        "u": rec.get("u", None),
        "served_risk": float(risk_true),
        "compliant": float(risk_true <= B_true),
        "hit": float(hit),
        "miss": float(miss),
        "recall": float(recall_at_k(items_take, rec["pos"], k=k)),
        "ndcg": float(ndcg_at_k(items_take, rec["pos"], k=k)),
        "proxy_utility": float(proxy_utility(np.asarray(rec["mu"]), take)),
        "bad_exposure": float((risk_true > B_true) and (hit == 0)),
        "weighted_bad_exposure": float(max(risk_true - B_true, 0.0) * miss),
        "risk_weighted_miss": float(risk_true * miss),
    }


def bisection_hard(mu, width_obs, B_true, k=10, max_iter=18, eps=1e-12):
    take0 = topk_idx(mu, k=k)
    if risk_of_take(width_obs, take0) <= B_true + eps:
        return take0
    if float(np.sort(width_obs)[:k].sum()) > B_true + eps:
        return take0

    def take_for(lam):
        return topk_idx(np.asarray(mu) - lam * np.asarray(width_obs), k=k)

    lam_lo, lam_hi = 0.0, 1.0
    take_hi = take_for(lam_hi)
    while risk_of_take(width_obs, take_hi) > B_true + eps and lam_hi < 1e6:
        lam_hi *= 2.0
        take_hi = take_for(lam_hi)
    best = take_hi
    for _ in range(max_iter):
        lam_mid = 0.5 * (lam_lo + lam_hi)
        take_mid = take_for(lam_mid)
        if risk_of_take(width_obs, take_mid) <= B_true + eps:
            best = take_mid
            lam_hi = lam_mid
        else:
            lam_lo = lam_mid
    return best


def quota_u_policy(records, B_true, tau, k=10, seed=2025, width_scale=1.0, cal_scale=1.0, warmup=WARMUP):
    N = len(records)
    V_total = int(np.floor((1.0 - tau) * N))
    V_left = V_total
    a0, b0 = 1.0, 1.0
    opp_seen, seen = 0, 0
    score_hist = []
    rng = np.random.default_rng(seed)
    out = []

    for rec in records:
        seen += 1
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)

        if risk_u_obs <= B_true:
            take = take_u
        else:
            take_c = bisection_hard(mu, width_obs, B_true, k=k)
            risk_c_obs = risk_of_take(width_obs, take_c)
            if risk_c_obs > B_true:
                take = take_u
                V_left -= 1
            else:
                opp_seen += 1
                du = proxy_utility(mu, take_u) - proxy_utility(mu, take_c)
                score_hist.append(float(du))

                remaining = max(1, N - seen)
                p_opp = (a0 + opp_seen) / (a0 + b0 + seen)
                exp_rem_opp = max(1.0, p_opp * remaining)
                desired_rate = min(1.0, V_left / exp_rem_opp) if exp_rem_opp > 0 else 0.0

                if V_left <= 0:
                    take = take_c
                else:
                    if len(score_hist) < int(warmup):
                        do_violate = rng.random() < desired_rate
                    else:
                        thr = float(np.quantile(np.asarray(score_hist), 1.0 - desired_rate))
                        do_violate = du >= thr
                    if do_violate and V_left > 0:
                        take = take_u
                        V_left -= 1
                    else:
                        take = take_c

        out.append(metrics_from_take(rec, take, B_true, k=k))
    return pd.DataFrame(out)


def build_rdu_old_scaler(records_val, B_true, k=10, width_scale=1.0, cal_scale=1.0):
    rows = []
    for rec in records_val:
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)
        if risk_u_obs <= B_true:
            continue
        take_c = bisection_hard(mu, width_obs, B_true, k=k)
        risk_c_obs = risk_of_take(width_obs, take_c)
        if risk_c_obs > B_true:
            continue
        du = proxy_utility(mu, take_u) - proxy_utility(mu, take_c)
        dr = max(0.0, risk_u_obs - risk_c_obs)
        rows.append((du, dr))

    if len(rows) == 0:
        return {"u_scale": 1.0, "r_scale": 1.0}

    arr = np.array(rows)
    return {
        "u_scale": float(np.median(np.abs(arr[:, 0] - np.median(arr[:, 0])))) + EPS,
        "r_scale": float(np.median(np.abs(arr[:, 1] - np.median(arr[:, 1])))) + EPS,
    }


def quota_rdu_old_policy(records, B_true, tau, lambda_val, scaler, k=10, seed=2025, width_scale=1.0, cal_scale=1.0, warmup=WARMUP):
    N = len(records)
    V_total = int(np.floor((1.0 - tau) * N))
    V_left = V_total
    a0, b0 = 1.0, 1.0
    opp_seen, seen = 0, 0
    score_hist = []
    rng = np.random.default_rng(seed)
    out = []

    for rec in records:
        seen += 1
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)

        if risk_u_obs <= B_true:
            take = take_u
        else:
            take_c = bisection_hard(mu, width_obs, B_true, k=k)
            risk_c_obs = risk_of_take(width_obs, take_c)

            if risk_c_obs > B_true:
                take = take_u
                V_left -= 1
            else:
                opp_seen += 1
                du = proxy_utility(mu, take_u) - proxy_utility(mu, take_c)
                dr = max(0.0, risk_u_obs - risk_c_obs)
                score = du / max(scaler["u_scale"], EPS) - float(lambda_val) * (dr / max(scaler["r_scale"], EPS))
                score_hist.append(float(score))

                remaining = max(1, N - seen)
                p_opp = (a0 + opp_seen) / (a0 + b0 + seen)
                exp_rem_opp = max(1.0, p_opp * remaining)
                desired_rate = min(1.0, V_left / exp_rem_opp) if exp_rem_opp > 0 else 0.0

                if V_left <= 0:
                    take = take_c
                else:
                    if len(score_hist) < int(warmup):
                        do_violate = rng.random() < desired_rate
                    else:
                        thr = float(np.quantile(np.asarray(score_hist), 1.0 - desired_rate))
                        do_violate = score >= thr
                    if do_violate and V_left > 0:
                        take = take_u
                        V_left -= 1
                    else:
                        take = take_c

        out.append(metrics_from_take(rec, take, B_true, k=k))
    return pd.DataFrame(out)


def wbe_threshold_policy(records, B_true, tau, gamma, k=10, seed=2025, width_scale=1.0, cal_scale=1.0, warmup=WARMUP):
    N = len(records)
    V_total = int(np.floor((1.0 - tau) * N))
    V_left = V_total
    a0, b0 = 1.0, 1.0
    opp_seen, seen = 0, 0
    score_hist = []
    rng = np.random.default_rng(seed)
    out = []

    for rec in records:
        seen += 1
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)

        if risk_u_obs <= B_true:
            take = take_u
        else:
            take_c = bisection_hard(mu, width_obs, B_true, k=k)
            risk_c_obs = risk_of_take(width_obs, take_c)
            if risk_c_obs > B_true:
                take = take_u
                V_left -= 1
            else:
                opp_seen += 1
                du = proxy_utility(mu, take_u) - proxy_utility(mu, take_c)
                exceed = max(risk_u_obs - B_true, 0.0)
                score = du - float(gamma) * exceed
                score_hist.append(float(score))

                remaining = max(1, N - seen)
                p_opp = (a0 + opp_seen) / (a0 + b0 + seen)
                exp_rem_opp = max(1.0, p_opp * remaining)
                desired_rate = min(1.0, V_left / exp_rem_opp) if exp_rem_opp > 0 else 0.0

                if V_left <= 0:
                    take = take_c
                else:
                    if len(score_hist) < int(warmup):
                        do_violate = rng.random() < desired_rate
                    else:
                        thr = float(np.quantile(np.asarray(score_hist), 1.0 - desired_rate))
                        do_violate = score >= thr
                    if do_violate and V_left > 0:
                        take = take_u
                        V_left -= 1
                    else:
                        take = take_c

        out.append(metrics_from_take(rec, take, B_true, k=k))
    return pd.DataFrame(out)


def penalty_policy(records, B_true, lam, k=10, width_scale=1.0, cal_scale=1.0):
    out = []
    for rec in records:
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take = topk_idx(mu - float(lam) * width_obs, k=k)
        out.append(metrics_from_take(rec, take, B_true, k=k))
    return pd.DataFrame(out)


def adaptive_penalty_policy(records, B_true, tau, lam0, eta0, k=10, width_scale=1.0, cal_scale=1.0):
    lam = float(lam0)
    out = []

    for t, rec in enumerate(records, start=1):
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take = topk_idx(mu - lam * width_obs, k=k)
        m = metrics_from_take(rec, take, B_true, k=k)
        out.append(m)
        viol = 1.0 - m["compliant"]
        lam = max(0.0, lam + (float(eta0) / np.sqrt(t)) * (viol - (1.0 - tau)))

    return pd.DataFrame(out)


def build_ewbe_table(records_val, B_true, k=10, width_scale=1.0, cal_scale=1.0):
    rows = []
    for rec in records_val:
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)
        if risk_u_obs <= B_true:
            continue
        take_c = bisection_hard(mu, width_obs, B_true, k=k)
        risk_c_obs = risk_of_take(width_obs, take_c)
        if risk_c_obs > B_true:
            continue

        util_u = proxy_utility(mu, take_u)
        util_c = proxy_utility(mu, take_c)
        du = util_u - util_c
        exceed = max(risk_u_obs - B_true, 0.0)
        slack = max(B_true - risk_c_obs, 0.0)
        dr = max(0.0, risk_u_obs - risk_c_obs)

        items_take_u = np.asarray(rec["items"])[take_u].tolist()
        miss_u = 1.0 - hit_at_k(items_take_u, rec["pos"], k=k)

        rows.append({
            "du": float(du),
            "util_u": float(util_u),
            "util_c": float(util_c),
            "exceed": float(exceed),
            "slack": float(slack),
            "dr": float(dr),
            "miss_u": float(miss_u),
            "ewbe_true": float(exceed * miss_u),
        })
    return pd.DataFrame(rows)


def _fit_logistic_or_constant(Xs, y):
    uniq = np.unique(y)
    if len(uniq) < 2:
        return {"type": "constant", "p": float(np.mean(y))}
    model = LogisticRegression(max_iter=1000)
    model.fit(Xs, y)
    return {"type": "logit", "model": model}


def _predict_from_bundle(bundle, Xs):
    if bundle["type"] == "constant":
        return np.full(Xs.shape[0], float(bundle["p"]), dtype=float)
    return bundle["model"].predict_proba(Xs)[:, 1]


def fit_ewbe_miss_model(tab):
    if not SKLEARN_OK:
        raise RuntimeError("scikit-learn is required for the expected-WBE proposal.")

    X = tab[["util_u", "exceed"]].to_numpy(dtype=float)
    med = np.median(X, axis=0)
    mad = np.median(np.abs(X - med), axis=0) + EPS
    Xs = (X - med) / mad
    y = tab["miss_u"].astype(int).to_numpy()

    bundle = _fit_logistic_or_constant(Xs, y)
    p_hat = _predict_from_bundle(bundle, Xs)
    sev = tab["exceed"].to_numpy(dtype=float) * p_hat

    scaler = {
        "x_median": med,
        "x_scale": mad,
        "u_scale": float(np.median(np.abs(tab["du"] - np.median(tab["du"])))) + EPS,
        "sev_scale": float(np.median(np.abs(sev - np.median(sev)))) + EPS,
    }
    return bundle, scaler


def predict_miss_prob(bundle, scaler, util_u, exceed):
    x = np.array([[util_u, exceed]], dtype=float)
    xs = (x - scaler["x_median"]) / scaler["x_scale"]
    return float(_predict_from_bundle(bundle, xs)[0])


def quota_rdu_ewbe_policy(records, B_true, tau, lambda_val, model_bundle, scaler, k=10, seed=2025, width_scale=1.0, cal_scale=1.0, warmup=WARMUP):
    N = len(records)
    V_total = int(np.floor((1.0 - tau) * N))
    V_left = V_total
    a0, b0 = 1.0, 1.0
    opp_seen, seen = 0, 0
    score_hist = []
    rng = np.random.default_rng(seed)
    out = []

    for rec in records:
        seen += 1
        mu = np.asarray(rec["mu"], dtype=float)
        width_obs = observed_width(rec, width_scale=width_scale, cal_scale=cal_scale)
        take_u = topk_idx(mu, k=k)
        risk_u_obs = risk_of_take(width_obs, take_u)

        if risk_u_obs <= B_true:
            take = take_u
        else:
            take_c = bisection_hard(mu, width_obs, B_true, k=k)
            risk_c_obs = risk_of_take(width_obs, take_c)

            if risk_c_obs > B_true:
                take = take_u
                V_left -= 1
            else:
                opp_seen += 1
                util_u = proxy_utility(mu, take_u)
                du = util_u - proxy_utility(mu, take_c)
                exceed = max(risk_u_obs - B_true, 0.0)
                p_hat = predict_miss_prob(model_bundle, scaler, util_u, exceed)
                sev = exceed * p_hat
                score = du / max(scaler["u_scale"], EPS) - float(lambda_val) * (sev / max(scaler["sev_scale"], EPS))
                score_hist.append(float(score))

                remaining = max(1, N - seen)
                p_opp = (a0 + opp_seen) / (a0 + b0 + seen)
                exp_rem_opp = max(1.0, p_opp * remaining)
                desired_rate = min(1.0, V_left / exp_rem_opp) if exp_rem_opp > 0 else 0.0

                if V_left <= 0:
                    take = take_c
                else:
                    if len(score_hist) < int(warmup):
                        do_violate = rng.random() < desired_rate
                    else:
                        thr = float(np.quantile(np.asarray(score_hist), 1.0 - desired_rate))
                        do_violate = score >= thr
                    if do_violate and V_left > 0:
                        take = take_u
                        V_left -= 1
                    else:
                        take = take_c

        out.append(metrics_from_take(rec, take, B_true, k=k))
    return pd.DataFrame(out)


def _safe_auc(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, score))


def _safe_ap(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    if len(np.unique(y)) < 2:
        return np.nan
    return float(average_precision_score(y, score))


def evaluate_predictive_checks(tab, model_bundle, scaler):
    p_hat = np.array(
        [predict_miss_prob(model_bundle, scaler, u, e) for u, e in zip(tab["util_u"], tab["exceed"])],
        dtype=float,
    )
    sev = tab["exceed"].to_numpy(dtype=float) * p_hat
    y = tab["miss_u"].astype(int).to_numpy()

    return pd.DataFrame([
        {
            "metric": "roc_auc",
            "exceed": _safe_auc(y, tab["exceed"].to_numpy()),
            "dr": _safe_auc(y, tab["dr"].to_numpy()),
            "expected_bad_exposure": _safe_auc(y, sev),
            "miss_probability": _safe_auc(y, p_hat),
        },
        {
            "metric": "avg_precision",
            "exceed": _safe_ap(y, tab["exceed"].to_numpy()),
            "dr": _safe_ap(y, tab["dr"].to_numpy()),
            "expected_bad_exposure": _safe_ap(y, sev),
            "miss_probability": _safe_ap(y, p_hat),
        },
    ])


def repeated_cv_incremental_slack(tab, seeds):
    X1 = tab[["util_u", "exceed"]].to_numpy(dtype=float)
    X2 = tab[["util_u", "exceed", "slack"]].to_numpy(dtype=float)
    y = tab["miss_u"].astype(int).to_numpy()
    rows = []

    for sd in seeds:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=int(sd))
        auc1, auc2, ap1, ap2 = [], [], [], []

        for tr, te in cv.split(X1, y):
            med1 = np.median(X1[tr], axis=0)
            mad1 = np.median(np.abs(X1[tr] - med1), axis=0) + EPS
            med2 = np.median(X2[tr], axis=0)
            mad2 = np.median(np.abs(X2[tr] - med2), axis=0) + EPS

            X1tr = (X1[tr] - med1) / mad1
            X1te = (X1[te] - med1) / mad1
            X2tr = (X2[tr] - med2) / mad2
            X2te = (X2[te] - med2) / mad2

            b1 = _fit_logistic_or_constant(X1tr, y[tr])
            b2 = _fit_logistic_or_constant(X2tr, y[tr])
            p1 = _predict_from_bundle(b1, X1te)
            p2 = _predict_from_bundle(b2, X2te)

            auc1.append(_safe_auc(y[te], p1))
            auc2.append(_safe_auc(y[te], p2))
            ap1.append(_safe_ap(y[te], p1))
            ap2.append(_safe_ap(y[te], p2))

        rows.append({
            "seed": sd,
            "auc_util_exceed": float(np.nanmean(auc1)),
            "auc_util_exceed_slack": float(np.nanmean(auc2)),
            "delta_auc_slack": float(np.nanmean(auc2) - np.nanmean(auc1)),
            "ap_util_exceed": float(np.nanmean(ap1)),
            "ap_util_exceed_slack": float(np.nanmean(ap2)),
            "delta_ap_slack": float(np.nanmean(ap2) - np.nanmean(ap1)),
        })

    per_seed = pd.DataFrame(rows)
    auc_mean, auc_lo, auc_hi, auc_sig = bootstrap_delta(
        per_seed["auc_util_exceed_slack"].to_numpy(),
        per_seed["auc_util_exceed"].to_numpy(),
        n_boot=N_BOOT,
        seed=2025,
    )
    ap_mean, ap_lo, ap_hi, ap_sig = bootstrap_delta(
        per_seed["ap_util_exceed_slack"].to_numpy(),
        per_seed["ap_util_exceed"].to_numpy(),
        n_boot=N_BOOT,
        seed=2025,
    )
    summary = pd.DataFrame([{
        "auc_delta_mean": auc_mean,
        "auc_ci_low": auc_lo,
        "auc_ci_high": auc_hi,
        "auc_significant": auc_sig,
        "ap_delta_mean": ap_mean,
        "ap_ci_low": ap_lo,
        "ap_ci_high": ap_hi,
        "ap_significant": ap_sig,
    }])
    return per_seed, summary


def choose_penalty_lam(val_records, B_true, tau, k=10, width_scale=1.0):
    rows = []
    for lam in PENALTY_LAM_GRID:
        df = penalty_policy(val_records, B_true, lam, k=k, width_scale=width_scale, cal_scale=1.0)
        rows.append({
            "lam": lam,
            "comp_dev": abs(float(df["compliant"].mean()) - tau),
            "wbe": float(df["weighted_bad_exposure"].mean()),
            "proxy_utility": float(df["proxy_utility"].mean()),
        })
    tab = pd.DataFrame(rows).sort_values(["comp_dev", "proxy_utility", "wbe"], ascending=[True, False, True])
    return float(tab.iloc[0]["lam"]), tab


def choose_adaptive_eta(val_records, B_true, tau, lam0, k=10, width_scale=1.0):
    rows = []
    for eta in ADAPTIVE_ETA_GRID:
        df = adaptive_penalty_policy(val_records, B_true, tau, lam0=lam0, eta0=eta, k=k, width_scale=width_scale, cal_scale=1.0)
        rows.append({
            "eta0": eta,
            "comp_dev": abs(float(df["compliant"].mean()) - tau),
            "wbe": float(df["weighted_bad_exposure"].mean()),
            "proxy_utility": float(df["proxy_utility"].mean()),
        })
    tab = pd.DataFrame(rows).sort_values(["comp_dev", "proxy_utility", "wbe"], ascending=[True, False, True])
    return float(tab.iloc[0]["eta0"]), tab


def choose_wbe_gamma(val_records, B_true, tau, k=10, width_scale=1.0):
    rows = []
    for gamma in GAMMA_GRID:
        df = wbe_threshold_policy(val_records, B_true, tau, gamma=gamma, k=k, seed=CTRL_SEEDS[0], width_scale=width_scale, cal_scale=1.0)
        rows.append({
            "gamma": gamma,
            "comp_dev": abs(float(df["compliant"].mean()) - tau),
            "wbe": float(df["weighted_bad_exposure"].mean()),
            "proxy_utility": float(df["proxy_utility"].mean()),
        })
    tab = pd.DataFrame(rows).sort_values(["comp_dev", "wbe", "proxy_utility"], ascending=[True, True, False])
    return float(tab.iloc[0]["gamma"]), tab


def choose_rdu_old(val_records, B_true, tau, k=10, width_scale=1.0):
    rows = []
    scaler = build_rdu_old_scaler(val_records, B_true, k=k, width_scale=width_scale, cal_scale=1.0)
    for lam in LAMBDA_GRID:
        df = quota_rdu_old_policy(
            val_records, B_true, tau, lambda_val=lam, scaler=scaler,
            k=k, seed=CTRL_SEEDS[0], width_scale=width_scale, cal_scale=1.0
        )
        rows.append({
            "lambda": lam,
            "comp_dev": abs(float(df["compliant"].mean()) - tau),
            "wbe": float(df["weighted_bad_exposure"].mean()),
            "ndcg": float(df["ndcg"].mean()),
            "proxy_utility": float(df["proxy_utility"].mean()),
            "u_scale": scaler["u_scale"],
            "r_scale": scaler["r_scale"],
        })

    tab = pd.DataFrame(rows)
    good = tab[tab["comp_dev"] <= 0.01].copy()
    if len(good) == 0:
        good = tab.sort_values(["comp_dev", "wbe", "proxy_utility"], ascending=[True, True, False]).copy()
    else:
        good = good.sort_values(["wbe", "proxy_utility"], ascending=[True, False]).copy()

    best = good.iloc[0]
    scaler = {"u_scale": float(best["u_scale"]), "r_scale": float(best["r_scale"])}
    return float(best["lambda"]), scaler, tab


def choose_rdu_ewbe(val_records, B_true, tau, k=10, width_scale=1.0):
    rows = []
    tab = build_ewbe_table(val_records, B_true, k=k, width_scale=width_scale, cal_scale=1.0)
    model_bundle, scaler = fit_ewbe_miss_model(tab)

    for lam in LAMBDA_GRID:
        df = quota_rdu_ewbe_policy(
            val_records, B_true, tau,
            lambda_val=lam,
            model_bundle=model_bundle,
            scaler=scaler,
            k=k,
            seed=CTRL_SEEDS[0],
            width_scale=width_scale,
            cal_scale=1.0,
        )
        rows.append({
            "lambda": lam,
            "comp_dev": abs(float(df["compliant"].mean()) - tau),
            "wbe": float(df["weighted_bad_exposure"].mean()),
            "ndcg": float(df["ndcg"].mean()),
            "proxy_utility": float(df["proxy_utility"].mean()),
            "u_scale": scaler["u_scale"],
            "sev_scale": scaler["sev_scale"],
            "n_opportunities": len(tab),
        })

    tab_res = pd.DataFrame(rows)
    good = tab_res[tab_res["comp_dev"] <= 0.01].copy()
    if len(good) == 0:
        good = tab_res.sort_values(["comp_dev", "wbe", "proxy_utility"], ascending=[True, True, False]).copy()
    else:
        good = good.sort_values(["wbe", "proxy_utility"], ascending=[True, False]).copy()

    best = good.iloc[0]
    return float(best["lambda"]), model_bundle, scaler, tab_res, tab


def fit_nominal_reference(val_records, B_true, tau, k=10):
    lam_nom, model_nom, scaler_nom, search_tab, fit_tab = choose_rdu_ewbe(
        val_records, B_true, tau, k=k, width_scale=1.0
    )

    df_ref = quota_rdu_ewbe_policy(
        val_records, B_true, tau,
        lambda_val=lam_nom,
        model_bundle=model_nom,
        scaler=scaler_nom,
        k=k,
        seed=CTRL_SEEDS[0],
        width_scale=1.0,
        cal_scale=1.0,
    )

    ref = {
        "compliance": float(df_ref["compliant"].mean()),
        "wbe": float(df_ref["weighted_bad_exposure"].mean()),
        "proxy": float(df_ref["proxy_utility"].mean()),
        "ndcg": float(df_ref["ndcg"].mean()),
    }
    return lam_nom, model_nom, scaler_nom, ref, search_tab, fit_tab


def exact_inverse_cal_scale(width_scale):
    return 1.0 / float(width_scale)


def summarize_method(dfm, ds_name, method_name, tau, k, width_scale, seed, quota_u_proxy_mean):
    return {
        "dataset": ds_name,
        "method": method_name,
        "tau": tau,
        "k": k,
        "width_scale": width_scale,
        "seed": seed,
        "compliance_mean": float(dfm["compliant"].mean()),
        "hit_mean": float(dfm["hit"].mean()),
        "recall_mean": float(dfm["recall"].mean()),
        "ndcg_mean": float(dfm["ndcg"].mean()),
        "bad_exposure_rate": float(dfm["bad_exposure"].mean()),
        "weighted_bad_exposure_mean": float(dfm["weighted_bad_exposure"].mean()),
        "risk_weighted_miss_mean": float(dfm["risk_weighted_miss"].mean()),
        "proxy_utility_mean": float(dfm["proxy_utility"].mean()),
        "proxy_gap_vs_quota_u_mean": float(quota_u_proxy_mean - dfm["proxy_utility"].mean()),
    }


if not SKLEARN_OK:
    raise RuntimeError("scikit-learn is required for this script but is not available.")


print("RUN_DIR   =", RUN_DIR)
print("OUT_DIR   =", OUT_DIR)
print("Using cand:", cand_path)
datasets = sorted(cand_df["dataset"].unique())
print("Datasets:", datasets)

# -----------------------------
# Validation diagnostics + main setting
# -----------------------------
diag_rows = []
diag_detail_rows = []
main_rows = []
main_runtime_rows = []
main_sig_rows = []
main_tuning_rows = []
lambda_rows = []

for ds_name in datasets:
    print(f"\n=== {ds_name} ===")
    cand_n = choose_largest_cand(ds_name)
    val_records = load_records(find_record_file(ds_name, "val", cand_n))
    test_records = load_records(find_record_file(ds_name, "test", cand_n))
    B_true, _ = resolve_budget(val_records, ds_name=ds_name, cand_n=cand_n, k_target=MAIN_K)

    ewbe_tab = build_ewbe_table(val_records, B_true, k=MAIN_K, width_scale=1.0, cal_scale=1.0)
    ewbe_model, ewbe_scaler = fit_ewbe_miss_model(ewbe_tab)

    pred_chk = evaluate_predictive_checks(ewbe_tab, ewbe_model, ewbe_scaler)
    pred_chk["dataset"] = ds_name
    diag_rows.extend(pred_chk.to_dict("records"))

    slack_seed_df, slack_sum_df = repeated_cv_incremental_slack(ewbe_tab, CV_SEEDS)
    slack_seed_df["dataset"] = ds_name
    slack_seed_df["kind"] = "per_seed"
    diag_detail_rows.extend(slack_seed_df.to_dict("records"))
    slack_sum_df["dataset"] = ds_name
    slack_sum_df["kind"] = "summary"
    diag_detail_rows.extend(slack_sum_df.to_dict("records"))

    lam_pen, _ = choose_penalty_lam(val_records, B_true, MAIN_TAU, k=MAIN_K, width_scale=1.0)
    eta_adp, _ = choose_adaptive_eta(val_records, B_true, MAIN_TAU, lam_pen, k=MAIN_K, width_scale=1.0)
    gamma_wbe, _ = choose_wbe_gamma(val_records, B_true, MAIN_TAU, k=MAIN_K, width_scale=1.0)
    lam_old, scaler_old, old_tab = choose_rdu_old(val_records, B_true, MAIN_TAU, k=MAIN_K, width_scale=1.0)
    lam_ewbe, ewbe_model_best, ewbe_scaler_best, ewbe_res_tab, ewbe_fit_tab = choose_rdu_ewbe(
        val_records, B_true, MAIN_TAU, k=MAIN_K, width_scale=1.0
    )

    main_tuning_rows.append({
        "dataset": ds_name,
        "tau": MAIN_TAU,
        "k": MAIN_K,
        "width_scale": 1.0,
        "penalty_lam": lam_pen,
        "adaptive_eta0": eta_adp,
        "wbe_gamma": gamma_wbe,
        "rdu_old_lambda": lam_old,
        "rdu_ewbe_lambda": lam_ewbe,
        "ewbe_fit_n": len(ewbe_fit_tab),
    })

    old_tab2 = old_tab.copy()
    old_tab2["dataset"] = ds_name
    old_tab2["variant"] = "RDU-Old"

    ewbe_res_tab2 = ewbe_res_tab.copy()
    ewbe_res_tab2["dataset"] = ds_name
    ewbe_res_tab2["variant"] = "RDU-EWBE"

    lambda_rows.extend(old_tab2.to_dict("records"))
    lambda_rows.extend(ewbe_res_tab2.to_dict("records"))

    pooled = defaultdict(list)
    for seed in CTRL_SEEDS:
        quota_u_proxy_mean = quota_u_policy(
            test_records, B_true, MAIN_TAU, k=MAIN_K, seed=seed, width_scale=1.0, cal_scale=1.0
        )["proxy_utility"].mean()

        methods = {
            "QuotaGate-U": timed_call(quota_u_policy, test_records, B_true, MAIN_TAU, MAIN_K, seed, 1.0, 1.0),
            "QuotaGate-RDU-Old": timed_call(quota_rdu_old_policy, test_records, B_true, MAIN_TAU, lam_old, scaler_old, MAIN_K, seed, 1.0, 1.0),
            "QuotaGate-RDU-EWBE": timed_call(quota_rdu_ewbe_policy, test_records, B_true, MAIN_TAU, lam_ewbe, ewbe_model_best, ewbe_scaler_best, MAIN_K, seed, 1.0, 1.0),
            "WBEThreshold": timed_call(wbe_threshold_policy, test_records, B_true, MAIN_TAU, gamma_wbe, MAIN_K, seed, 1.0, 1.0),
            "AdaptivePenalty": timed_call(adaptive_penalty_policy, test_records, B_true, MAIN_TAU, lam_pen, eta_adp, MAIN_K, 1.0, 1.0),
            "Penalty": timed_call(penalty_policy, test_records, B_true, lam_pen, MAIN_K, 1.0, 1.0),
        }

        for m, (dfm, sec) in methods.items():
            main_rows.append(summarize_method(dfm, ds_name, m, MAIN_TAU, MAIN_K, 1.0, seed, quota_u_proxy_mean))
            main_runtime_rows.append({
                "dataset": ds_name,
                "method": m,
                "tau": MAIN_TAU,
                "k": MAIN_K,
                "width_scale": 1.0,
                "seed": seed,
                "ms_per_user": 1000.0 * sec / len(test_records),
                "sec_total": sec,
            })
            pooled[m].append(dfm)

    pooled = {m: pd.concat(v, ignore_index=True) for m, v in pooled.items()}

    for comp in ["QuotaGate-RDU-Old", "WBEThreshold"]:
        mean_, lo_, hi_, sig = bootstrap_delta(
            pooled[comp]["weighted_bad_exposure"].to_numpy(),
            pooled["QuotaGate-RDU-EWBE"]["weighted_bad_exposure"].to_numpy(),
            n_boot=N_BOOT,
            seed=2025,
        )
        main_sig_rows.append({
            "dataset": ds_name,
            "comparison": f"QuotaGate-RDU-EWBE vs {comp}",
            "metric": "weighted_bad_exposure",
            "delta_mean": mean_,
            "ci_low": lo_,
            "ci_high": hi_,
            "significant": sig,
        })

        for metric in ["compliant", "ndcg", "proxy_utility"]:
            mean_, lo_, hi_, sig = bootstrap_delta(
                pooled["QuotaGate-RDU-EWBE"][metric].to_numpy(),
                pooled[comp][metric].to_numpy(),
                n_boot=N_BOOT,
                seed=2025,
            )
            main_sig_rows.append({
                "dataset": ds_name,
                "comparison": f"QuotaGate-RDU-EWBE vs {comp}",
                "metric": metric,
                "delta_mean": mean_,
                "ci_low": lo_,
                "ci_high": hi_,
                "significant": sig,
            })

diag_df = pd.DataFrame(diag_rows)
diag_detail_df = pd.DataFrame(diag_detail_rows)
main_df = pd.DataFrame(main_rows)
main_runtime_df = pd.DataFrame(main_runtime_rows)
main_sig_df = pd.DataFrame(main_sig_rows)
main_tuning_df = pd.DataFrame(main_tuning_rows)
lambda_df = pd.DataFrame(lambda_rows)

main_summary_df = agg_mean_std(
    main_df,
    ["dataset", "method", "tau", "k", "width_scale"],
    [
        "compliance_mean",
        "hit_mean",
        "recall_mean",
        "ndcg_mean",
        "bad_exposure_rate",
        "weighted_bad_exposure_mean",
        "risk_weighted_miss_mean",
        "proxy_utility_mean",
        "proxy_gap_vs_quota_u_mean",
    ],
)
main_runtime_summary_df = agg_mean_std(
    main_runtime_df,
    ["dataset", "method", "tau", "k", "width_scale"],
    ["ms_per_user", "sec_total"],
)

save_df(diag_df, "ewbe_inverse_validation_predictive_checks.csv")
save_df(diag_detail_df, "ewbe_inverse_validation_slack_incremental_value.csv")
save_df(main_df, "ewbe_inverse_main_seed_level.csv")
save_df(main_summary_df, "ewbe_inverse_main_summary.csv")
save_df(main_runtime_summary_df, "ewbe_inverse_main_runtime_summary.csv")
save_df(main_sig_df, "ewbe_inverse_main_significance.csv")
save_df(main_tuning_df, "ewbe_inverse_main_tuning_table.csv")
save_df(lambda_df, "ewbe_inverse_lambda_sweeps.csv")

# Main pass rule
pass_rows = []
n_wbe_tiebeat = 0
all_comp_ok = True
all_ndcg_ok = True

for ds_name in datasets:
    sub = main_summary_df[main_summary_df["dataset"] == ds_name].copy()
    re = sub[sub["method"] == "QuotaGate-RDU-EWBE"].iloc[0]
    wt = sub[sub["method"] == "WBEThreshold"].iloc[0]
    ro = sub[sub["method"] == "QuotaGate-RDU-Old"].iloc[0]

    tiebeat = bool(re["weighted_bad_exposure_mean"] <= wt["weighted_bad_exposure_mean"] + 1e-12)
    comp_ok = bool(abs(float(re["compliance_mean"]) - MAIN_TAU) <= 0.01)
    ndcg_ok = bool(float(re["ndcg_mean"]) >= float(ro["ndcg_mean"]) - 0.002)

    n_wbe_tiebeat += int(tiebeat)
    all_comp_ok = all_comp_ok and comp_ok
    all_ndcg_ok = all_ndcg_ok and ndcg_ok

    pass_rows.append({
        "dataset": ds_name,
        "ewbe_rdu_wbe": float(re["weighted_bad_exposure_mean"]),
        "wbe_threshold_wbe": float(wt["weighted_bad_exposure_mean"]),
        "ewbe_rdu_compliance": float(re["compliance_mean"]),
        "ewbe_rdu_ndcg": float(re["ndcg_mean"]),
        "old_rdu_ndcg": float(ro["ndcg_mean"]),
        "tie_or_beat_wbe_threshold": tiebeat,
        "compliance_ok": comp_ok,
        "ndcg_not_visibly_worse_than_old": ndcg_ok,
    })

PASS_MAIN = bool((n_wbe_tiebeat >= 2) and all_comp_ok and all_ndcg_ok)

pass_df = pd.DataFrame(pass_rows)
pass_df["pass_rule_overall"] = PASS_MAIN
save_df(pass_df, "ewbe_inverse_main_pass_check.csv")

with open(os.path.join(OUT_SNIPS, "ewbe_inverse_pass_report.txt"), "w") as f:
    f.write(f"PASS_MAIN={PASS_MAIN}\n")
    f.write(f"datasets_tie_or_beat_wbe_threshold={n_wbe_tiebeat}/3\n")
    f.write(f"all_comp_ok={all_comp_ok}\n")
    f.write(f"all_ndcg_ok={all_ndcg_ok}\n")

print("PASS_MAIN =", PASS_MAIN)

# -----------------------------
# Robustness with exact inverse transport
# -----------------------------
robust_rows = []
robust_runtime_rows = []
robust_sig_rows = []
robust_tuning_rows = []
recovery_rows = []

if PASS_MAIN or FORCE_ROBUSTNESS:
    print("Running robustness pack with exact inverse transport calibration...")

    for ds_name in datasets:
        cand_n = choose_largest_cand(ds_name)
        val_records = load_records(find_record_file(ds_name, "val", cand_n))
        test_records = load_records(find_record_file(ds_name, "test", cand_n))
        B_true, _ = resolve_budget(val_records, ds_name=ds_name, cand_n=cand_n, k_target=MAIN_K)

        for tau in ROBUST_TAUS:
            lam_pen, _ = choose_penalty_lam(val_records, B_true, tau, k=MAIN_K, width_scale=1.0)
            eta_adp, _ = choose_adaptive_eta(val_records, B_true, tau, lam_pen, k=MAIN_K, width_scale=1.0)

            lam_nom, ewbe_model_nom, ewbe_scaler_nom, ref, nominal_search_tab, nominal_fit_tab = fit_nominal_reference(
                val_records, B_true, tau, k=MAIN_K
            )

            for width_scale in ROBUST_WIDTH_SCALES:
                cal_scale = exact_inverse_cal_scale(width_scale)
                eff_product = float(width_scale) * float(cal_scale)

                robust_tuning_rows.append({
                    "dataset": ds_name,
                    "tau": tau,
                    "k": MAIN_K,
                    "width_scale": width_scale,
                    "exact_inverse_cal_scale": cal_scale,
                    "effective_width_product": eff_product,
                    "penalty_lam": lam_pen,
                    "adaptive_eta0": eta_adp,
                    "ewbe_lambda_nominal": lam_nom,
                    "ref_compliance": ref["compliance"],
                    "ref_wbe": ref["wbe"],
                    "ref_proxy_utility": ref["proxy"],
                    "ref_ndcg": ref["ndcg"],
                    "nominal_fit_n": len(nominal_fit_tab),
                })

                pooled_raw = []
                pooled_cal = []
                pooled_nominal = []

                for seed in CTRL_SEEDS:
                    nominal_df, sec_nom = timed_call(
                        quota_rdu_ewbe_policy,
                        test_records, B_true, tau,
                        lam_nom, ewbe_model_nom, ewbe_scaler_nom,
                        MAIN_K, seed, 1.0, 1.0
                    )
                    quota_u_proxy_mean = quota_u_policy(
                        test_records, B_true, tau, k=MAIN_K, seed=seed, width_scale=width_scale, cal_scale=1.0
                    )["proxy_utility"].mean()

                    methods = {
                        "QuotaGate-U": timed_call(quota_u_policy, test_records, B_true, tau, MAIN_K, seed, width_scale, 1.0),
                        "QuotaGate-RDU-EWBE": timed_call(quota_rdu_ewbe_policy, test_records, B_true, tau, lam_nom, ewbe_model_nom, ewbe_scaler_nom, MAIN_K, seed, width_scale, 1.0),
                        "QuotaGate-RDU-EWBE-Cal": timed_call(quota_rdu_ewbe_policy, test_records, B_true, tau, lam_nom, ewbe_model_nom, ewbe_scaler_nom, MAIN_K, seed, width_scale, cal_scale),
                        "AdaptivePenalty": timed_call(adaptive_penalty_policy, test_records, B_true, tau, lam_pen, eta_adp, MAIN_K, width_scale, 1.0),
                    }

                    for m, (dfm, sec) in methods.items():
                        robust_rows.append(summarize_method(dfm, ds_name, m, tau, MAIN_K, width_scale, seed, quota_u_proxy_mean))
                        robust_runtime_rows.append({
                            "dataset": ds_name,
                            "method": m,
                            "tau": tau,
                            "k": MAIN_K,
                            "width_scale": width_scale,
                            "seed": seed,
                            "ms_per_user": 1000.0 * sec / len(test_records),
                            "sec_total": sec,
                        })

                    pooled_nominal.append(nominal_df)
                    pooled_raw.append(methods["QuotaGate-RDU-EWBE"][0])
                    pooled_cal.append(methods["QuotaGate-RDU-EWBE-Cal"][0])

                pooled_nominal = pd.concat(pooled_nominal, ignore_index=True)
                pooled_raw = pd.concat(pooled_raw, ignore_index=True)
                pooled_cal = pd.concat(pooled_cal, ignore_index=True)

                # raw vs cal
                mean_, lo_, hi_, sig = bootstrap_delta(
                    pooled_raw["weighted_bad_exposure"].to_numpy(),
                    pooled_cal["weighted_bad_exposure"].to_numpy(),
                    n_boot=N_BOOT,
                    seed=2025,
                )
                robust_sig_rows.append({
                    "dataset": ds_name,
                    "tau": tau,
                    "width_scale": width_scale,
                    "comparison": "QuotaGate-RDU-EWBE-Cal vs QuotaGate-RDU-EWBE",
                    "metric": "weighted_bad_exposure",
                    "delta_mean": mean_,
                    "ci_low": lo_,
                    "ci_high": hi_,
                    "significant": sig,
                })

                for metric in ["compliant", "ndcg", "proxy_utility"]:
                    mean_, lo_, hi_, sig = bootstrap_delta(
                        pooled_cal[metric].to_numpy(),
                        pooled_raw[metric].to_numpy(),
                        n_boot=N_BOOT,
                        seed=2025,
                    )
                    robust_sig_rows.append({
                        "dataset": ds_name,
                        "tau": tau,
                        "width_scale": width_scale,
                        "comparison": "QuotaGate-RDU-EWBE-Cal vs QuotaGate-RDU-EWBE",
                        "metric": metric,
                        "delta_mean": mean_,
                        "ci_low": lo_,
                        "ci_high": hi_,
                        "significant": sig,
                    })

                # calibrated vs nominal recovery audit
                for metric in ["compliant", "weighted_bad_exposure", "ndcg", "proxy_utility"]:
                    mean_, lo_, hi_, sig = bootstrap_delta(
                        pooled_cal[metric].to_numpy(),
                        pooled_nominal[metric].to_numpy(),
                        n_boot=N_BOOT,
                        seed=2025,
                    )
                    recovery_rows.append({
                        "dataset": ds_name,
                        "tau": tau,
                        "width_scale": width_scale,
                        "exact_inverse_cal_scale": cal_scale,
                        "effective_width_product": eff_product,
                        "comparison": "QuotaGate-RDU-EWBE-Cal vs Nominal",
                        "metric": metric,
                        "delta_mean": mean_,
                        "ci_low": lo_,
                        "ci_high": hi_,
                        "significant": sig,
                        "nominal_mean": float(pooled_nominal[metric].mean()),
                        "calibrated_mean": float(pooled_cal[metric].mean()),
                    })

robust_df = pd.DataFrame(robust_rows)
robust_runtime_df = pd.DataFrame(robust_runtime_rows)
robust_sig_df = pd.DataFrame(robust_sig_rows)
robust_tuning_df = pd.DataFrame(robust_tuning_rows)
recovery_df = pd.DataFrame(recovery_rows)

if len(robust_df):
    robust_summary_df = agg_mean_std(
        robust_df,
        ["dataset", "method", "tau", "k", "width_scale"],
        [
            "compliance_mean",
            "hit_mean",
            "recall_mean",
            "ndcg_mean",
            "bad_exposure_rate",
            "weighted_bad_exposure_mean",
            "risk_weighted_miss_mean",
            "proxy_utility_mean",
            "proxy_gap_vs_quota_u_mean",
        ],
    )
    robust_runtime_summary_df = agg_mean_std(
        robust_runtime_df,
        ["dataset", "method", "tau", "k", "width_scale"],
        ["ms_per_user", "sec_total"],
    )
    save_df(robust_df, "ewbe_inverse_robustness_seed_level.csv")
    save_df(robust_summary_df, "ewbe_inverse_robustness_summary.csv")
    save_df(robust_runtime_summary_df, "ewbe_inverse_robustness_runtime_summary.csv")
    save_df(robust_sig_df, "ewbe_inverse_robustness_significance.csv")
    save_df(robust_tuning_df, "ewbe_inverse_robustness_tuning_table.csv")
    save_df(recovery_df, "ewbe_inverse_nominal_recovery_audit.csv")
else:
    robust_summary_df = pd.DataFrame()

# -----------------------------
# Figures
# -----------------------------
# Main frontier
fig, axes = plt.subplots(1, len(datasets), figsize=(5.2 * len(datasets), 4.4), squeeze=False)
for c, ds_name in enumerate(datasets):
    sub = main_summary_df[
        (main_summary_df["dataset"] == ds_name)
        & (np.isclose(main_summary_df["tau"], MAIN_TAU))
        & (main_summary_df["k"] == MAIN_K)
    ].copy()
    ax = axes[0, c]
    order = ["QuotaGate-U", "QuotaGate-RDU-Old", "QuotaGate-RDU-EWBE", "WBEThreshold", "AdaptivePenalty", "Penalty"]
    sub["method"] = pd.Categorical(sub["method"], categories=order, ordered=True)
    sub = sub.sort_values("method")
    for _, row in sub.iterrows():
        ax.errorbar(
            row["weighted_bad_exposure_mean"],
            row["proxy_gap_vs_quota_u_mean"],
            xerr=row["weighted_bad_exposure_mean_std"],
            yerr=row["proxy_gap_vs_quota_u_mean_std"],
            fmt="o",
            capsize=3,
        )
        ax.annotate(
            str(row["method"]),
            (row["weighted_bad_exposure_mean"], row["proxy_gap_vs_quota_u_mean"]),
            textcoords="offset points",
            xytext=(4, 4),
            fontsize=8,
        )
    ax.set_title(ds_name)
    ax.set_xlabel("Weighted bad exposure")
    ax.set_ylabel("Proxy-gap vs QuotaGate-U")
    ax.grid(True, alpha=0.25)
fig.suptitle("Expected-WBE RDU at the decisive main setting", y=1.02, fontsize=13)
fig.tight_layout()
fig_path = os.path.join(OUT_FIGS, "fig_ewbe_inverse_main_frontier_with_seed_errorbars.png")
fig.savefig(fig_path, bbox_inches="tight", dpi=300)
print("Saved:", fig_path)
plt.show()

# Lambda frontier
fig, axes = plt.subplots(1, len(datasets), figsize=(5.2 * len(datasets), 4.4), squeeze=False)
for c, ds_name in enumerate(datasets):
    sub = lambda_df[lambda_df["dataset"] == ds_name].copy()
    ax = axes[0, c]
    for variant in ["RDU-Old", "RDU-EWBE"]:
        s2 = sub[sub["variant"] == variant].sort_values("lambda")
        ax.plot(s2["proxy_utility"], s2["wbe"], marker="o", linewidth=2.0, label=variant)
    ax.set_title(ds_name)
    ax.set_xlabel("Validation proxy utility")
    ax.set_ylabel("Validation WBE")
    ax.grid(True, alpha=0.25)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Old vs expected-WBE RDU frontier on validation", y=1.02, fontsize=13)
fig.tight_layout()
fig_path = os.path.join(OUT_FIGS, "fig_ewbe_inverse_lambda_frontier.png")
fig.savefig(fig_path, bbox_inches="tight", dpi=300)
print("Saved:", fig_path)
plt.show()

# Validation diagnostics
if len(diag_df):
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), squeeze=False)
    for j, metric_name in enumerate(["roc_auc", "avg_precision"]):
        ax = axes[0, j]
        sub = diag_df[diag_df["metric"] == metric_name].copy()
        x = np.arange(len(sub))
        w = 0.18
        ax.bar(x - 1.5 * w, sub["exceed"].values, width=w, label="Exceed")
        ax.bar(x - 0.5 * w, sub["dr"].values, width=w, label="dR")
        ax.bar(x + 0.5 * w, sub["expected_bad_exposure"].values, width=w, label="Exceed * p_hat")
        ax.bar(x + 1.5 * w, sub["miss_probability"].values, width=w, label="p_hat")
        ax.set_xticks(x, sub["dataset"].values)
        ax.set_title(metric_name)
        ax.grid(True, axis="y", alpha=0.25)
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle("Validation diagnostics for the expected-WBE proposal", y=1.02, fontsize=13)
    fig.tight_layout()
    fig_path = os.path.join(OUT_FIGS, "fig_ewbe_inverse_validation_diagnostics.png")
    fig.savefig(fig_path, bbox_inches="tight", dpi=300)
    print("Saved:", fig_path)
    plt.show()

# Exact inverse repair figure
if len(robust_summary_df):
    fig, axes = plt.subplots(1, len(datasets), figsize=(5.2 * len(datasets), 4.4), squeeze=False)
    for c, ds_name in enumerate(datasets):
        sub = robust_summary_df[
            (robust_summary_df["dataset"] == ds_name)
            & (np.isclose(robust_summary_df["tau"], 0.90))
            & (robust_summary_df["k"] == MAIN_K)
        ].copy()
        ax = axes[0, c]
        for method in ["QuotaGate-RDU-EWBE", "QuotaGate-RDU-EWBE-Cal", "AdaptivePenalty"]:
            s2 = sub[sub["method"] == method].sort_values("width_scale")
            ax.plot(s2["width_scale"], s2["weighted_bad_exposure_mean"], marker="o", linewidth=2.0, label=method)
        ax.set_title(ds_name)
        ax.set_xlabel("Observed width scale")
        ax.set_ylabel("Weighted bad exposure")
        ax.grid(True, alpha=0.25)
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle("Exact inverse transport repair at tau=0.90", y=1.02, fontsize=13)
    fig.tight_layout()
    fig_path = os.path.join(OUT_FIGS, "fig_ewbe_inverse_miscalibration_repair_tau09.png")
    fig.savefig(fig_path, bbox_inches="tight", dpi=300)
    print("Saved:", fig_path)
    plt.show()

    # Nominal recovery figure
    fig, axes = plt.subplots(1, len(datasets), figsize=(5.2 * len(datasets), 4.4), squeeze=False)
    for c, ds_name in enumerate(datasets):
        sub = recovery_df[
            (recovery_df["dataset"] == ds_name)
            & (np.isclose(recovery_df["tau"], 0.90))
            & (recovery_df["metric"] == "weighted_bad_exposure")
        ].copy().sort_values("width_scale")
        ax = axes[0, c]
        ax.plot(sub["width_scale"], sub["calibrated_mean"], marker="o", linewidth=2.0, label="Calibrated")
        ax.plot(sub["width_scale"], sub["nominal_mean"], marker="s", linewidth=2.0, label="Nominal ref")
        ax.set_title(ds_name)
        ax.set_xlabel("Observed width scale")
        ax.set_ylabel("WBE mean")
        ax.grid(True, alpha=0.25)
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle("Nominal recovery audit under exact inverse transport", y=1.02, fontsize=13)
    fig.tight_layout()
    fig_path = os.path.join(OUT_FIGS, "fig_ewbe_inverse_nominal_recovery_tau09.png")
    fig.savefig(fig_path, bbox_inches="tight", dpi=300)
    print("Saved:", fig_path)
    plt.show()

# -----------------------------
# README / package
# -----------------------------
readme = textwrap.dedent(f"""
Expected-WBE RDU round with exact inverse transport calibration.

Core idea
---------
Keep the same EWBE controller:
    sev = exceed * p_hat

Main setting
------------
Still tuned at width_scale = 1.0.

Exact robustness fix
--------------------
Under the synthetic global multiplicative mismatch family:
    w_eff = width_scale * cal_scale * w

Exact recovery requires:
    width_scale * cal_scale = 1

So this script uses:
    cal_scale = 1 / width_scale

with the nominal controller frozen:
    - lambda*
    - miss model
    - feature scales
    - severity scale

Main outputs
------------
- tables/ewbe_inverse_main_summary.csv
- tables/ewbe_inverse_main_significance.csv
- tables/ewbe_inverse_main_tuning_table.csv
- tables/ewbe_inverse_main_pass_check.csv
- tables/ewbe_inverse_lambda_sweeps.csv
- figs/fig_ewbe_inverse_main_frontier_with_seed_errorbars.png
- figs/fig_ewbe_inverse_lambda_frontier.png
- figs/fig_ewbe_inverse_validation_diagnostics.png

Robustness outputs
------------------
- tables/ewbe_inverse_robustness_summary.csv
- tables/ewbe_inverse_robustness_significance.csv
- tables/ewbe_inverse_robustness_tuning_table.csv
- tables/ewbe_inverse_nominal_recovery_audit.csv
- figs/fig_ewbe_inverse_miscalibration_repair_tau09.png
- figs/fig_ewbe_inverse_nominal_recovery_tau09.png

Detached run from a notebook cell
---------------------------------
!tmux new-session -d -s ewbeinverse \\
"bash -lc 'python -u exact_inverse_round.py 2>&1 | tee {os.path.join(RUN_DIR, 'expected_wbe_exact_inverse_round.log')}'"
""").strip()

with open(os.path.join(OUT_DIR, "README.md"), "w") as f:
    f.write(readme + "\n")
with open(os.path.join(OUT_SNIPS, "tmux_run_instructions.txt"), "w") as f:
    f.write(readme + "\n")

manifest = []
for root, _, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        manifest.append({"path": os.path.relpath(p, OUT_DIR), "bytes": os.path.getsize(p)})
pd.DataFrame(manifest).sort_values("path").to_csv(os.path.join(OUT_DIR, "manifest.csv"), index=False)

zip_path = OUT_DIR + ".zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)

print("\nDONE.")
print("Output folder:", OUT_DIR)
print("Zip package :", zip_path)


## Inspect the most important generated artifacts

This cell prints the key tables and points you to the generated figures.


In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = PROJECT_ROOT / 'expected_wbe_exact_inverse_round'
assert OUT_DIR.exists(), f'Expected output folder not found: {OUT_DIR}'

for rel in [
    'tables/ewbe_inverse_main_summary.csv',
    'tables/ewbe_inverse_main_significance.csv',
    'tables/ewbe_inverse_robustness_summary.csv',
    'tables/ewbe_inverse_nominal_recovery_audit.csv',
]:
    p = OUT_DIR / rel
    print('
===', rel, '===')
    display(pd.read_csv(p).head(20))

print('
Generated figures:')
for rel in [
    'figs/fig_ewbe_inverse_main_frontier_with_seed_errorbars.png',
    'figs/fig_ewbe_inverse_lambda_frontier.png',
    'figs/fig_ewbe_inverse_validation_diagnostics.png',
    'figs/fig_ewbe_inverse_miscalibration_repair_tau09.png',
    'figs/fig_ewbe_inverse_nominal_recovery_tau09.png',
]:
    p = OUT_DIR / rel
    print(' -', p)

## Optional: inspect older supporting evidence already packaged beside the notebook

These cells are **read-only helpers**. They are not required to rerun the final line, but they are convenient for browsing the older base-controller audit and semantic-bridge evidence that still support the final paper claims.

In [ ]:
from pathlib import Path
import pandas as pd

support_root = PROJECT_ROOT / 'final_submission_pack_final_gap_closer'
if support_root.exists():
    print('Found:', support_root)
    for rel in [
        'tables/feedback_claims_bulletproof.csv',
        'tables/finalgap_static_summary.csv',
        'tables/risk_quality_correlations.csv',
        'tables/uncertainty_realized_validation.csv',
    ]:
        p = support_root / rel
        if p.exists():
            print('
===', rel, '===')
            display(pd.read_csv(p).head(20))
else:
    print('Optional support folder not present:', support_root)

## Summary for reviewers

This curated notebook intentionally removes the large exploratory history so that reviewers can:
1. point the notebook to a local `PROJECT_ROOT`,
2. verify that `tables/` and `records/` are present,
3. run the exact-inverse final line once,
4. inspect the final paper tables and figures directly.

The original notebook remains useful as a lab-history artifact, but this curated notebook is the one optimized for **easy, final-line reproducibility**.